# Cebuano Sentiment Analysis — XLM-RoBERTa Backbone
**3-Class SA-AoA with Dependency Graph  |  xlm-roberta-base**

## 1. Setup

In [ ]:
# Kaggle setup - no drive mount needed
import os

# Check we're on Kaggle
if os.path.exists('/kaggle'):
    print("Running on Kaggle")
    print(f"GPU: {os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()}")
else:
    print("WARNING: Not running on Kaggle - paths may need adjustment")

# List input datasets
input_dir = "/kaggle/input"
if os.path.exists(input_dir):
    for d in os.listdir(input_dir):
        files = os.listdir(os.path.join(input_dir, d))
        print(f"  Dataset '{d}': {files}")


In [ ]:
!pip install -q transformers torch scikit-learn pandas matplotlib seaborn
print("Done")

In [ ]:
import json
import random
import time
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from torch.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Configuration

In [ ]:
# ==================== KAGGLE PATHS ====================
import glob, os

json_matches = glob.glob("/kaggle/input/**/*.json", recursive=True) + \
               glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
if json_matches:
    DATA_PATH = json_matches[0]
    print(f"Auto-detected dataset: {DATA_PATH}")
else:
    DATA_PATH = "/kaggle/input/licr-dataset200k/LICR_Cebert_Dataset.json"
    print(f"Using default path: {DATA_PATH}")

MODEL_NAME = "xlm-roberta-base"
OUTPUT_DIR = "/kaggle/working/checkpoints_xlmroberta"

# Bump when graph builder logic changes → forces cache rebuild
CACHE_VERSION = "v3-xlmroberta"

# XLM-RoBERTa uses a 12-layer 768-dim RoBERTa encoder pre-trained
# on 2.5TB of filtered CommonCrawl data across 100 languages,
# including Cebuano/Filipino. It has a 250k-token SentencePiece vocab
# which gives better Cebuano sub-word coverage than mBERT.
# ENCODER_LR is lowered to 5e-6 — XLM-R is sensitive to large LR.
@dataclass
class Config:
    DATA_PATH: str = DATA_PATH
    MODEL_NAME: str = MODEL_NAME
    OUTPUT_DIR: str = OUTPUT_DIR
    CACHE_DIR: str = f"/kaggle/working/cache_{CACHE_VERSION}"
    CACHE_VERSION: str = CACHE_VERSION

    MAX_LEN: int = 32
    NUM_CLASSES: int = 3
    HIDDEN_DIM: int = 768
    DROPOUT: float = 0.3
    CLS_DROPOUT: float = 0.3

    NUM_EDGE_TYPES: int = 7
    GRAPH_HIDDEN: int = 256
    NUM_GAT_HEADS: int = 4
    GRAPH_DROPOUT: float = 0.2

    # ── UPGRADE 1: Maximize Hardware Utility ──────────────────────────────
    # Increased from 16 → 32 to better saturate the T4 GPU pipeline.
    BATCH_SIZE: int = 32
    EPOCHS: int = 10
    LEARNING_RATE: float = 2e-5
    ENCODER_LR: float = 5e-6
    WEIGHT_DECAY: float = 0.01
    WARMUP_RATIO: float = 0.1
    GRADIENT_CLIP: float = 1.0
    # Simulates an effective batch size of 32 × 4 = 128 without OOM.
    GRAD_ACCUM_STEPS: int = 4

    # Loss weights
    ALPHA_SCL: float = 0.3
    LAMBDA_GRAPH_AUX: float = 0.5   # graph branch supervision (was 0.2)
    LABEL_SMOOTHING: float = 0.1
    TEMPERATURE_SCL: float = 0.07

    # Entropy schedule
    GAMMA_ENTROPY_START: float = 0.20
    GAMMA_ENTROPY_END:   float = 0.05
    ENTROPY_WARMUP_EPOCHS: int = 4   # hold high for first 4 of 10

    # Graph bias scaling — active in BOTH training AND eval
    GRAPH_BIAS_ALPHA: float = 4.0

    # CLS masking schedule (prevents CLS bypass of graph branch)
    CLS_MASK_START: float = 0.5    # mask CLS 50% early
    CLS_MASK_END: float = 0.0      # no masking at end

    # Caching
    USE_CACHE: bool = True
    FORCE_REBUILD_CACHE: bool = False
    CHUNK_SIZE: int = 5000

    # ── UPGRADE 3: Ablation Study Toggles ─────────────────────────────────
    # Set to False individually to isolate the contribution of each component.
    ENABLE_SA_AOA: bool = True   # If False, bypasses AoA and uses mean(syntax_states)
    ENABLE_ENTROPY: bool = True  # If False, skips the Entropy Regularization loss term
    ENABLE_SCL: bool = True      # If False, skips the Supervised Contrastive loss term

    CLASS_WEIGHTS: Optional[torch.Tensor] = None

config = Config()
Path(config.CACHE_DIR).mkdir(exist_ok=True, parents=True)
Path(config.OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

print(f"{'='*65}")
print(f"EPOCHS = {config.EPOCHS}")
print(f"BATCH_SIZE = {config.BATCH_SIZE}  (GRAD_ACCUM_STEPS = {config.GRAD_ACCUM_STEPS}  →  effective = {config.BATCH_SIZE * config.GRAD_ACCUM_STEPS})")
print(f"GRAPH_BIAS_ALPHA = {config.GRAPH_BIAS_ALPHA}")
print(f"LAMBDA_GRAPH_AUX = {config.LAMBDA_GRAPH_AUX}")
print(f"CLS masking: {config.CLS_MASK_START} -> {config.CLS_MASK_END}")
print(f"Entropy: {config.GAMMA_ENTROPY_START} -> {config.GAMMA_ENTROPY_END} (warmup {config.ENTROPY_WARMUP_EPOCHS} ep)")
print(f"CACHE_VERSION = {CACHE_VERSION}")
print(f"Data: {config.DATA_PATH}")
print(f"Model: {config.MODEL_NAME}")
print(f"--- Ablation Toggles ---")
print(f"  ENABLE_SA_AOA  = {config.ENABLE_SA_AOA}")
print(f"  ENABLE_ENTROPY = {config.ENABLE_ENTROPY}")
print(f"  ENABLE_SCL     = {config.ENABLE_SCL}")
print(f"{'='*65}")
assert os.path.exists(config.DATA_PATH), f"Dataset not found at {config.DATA_PATH}"


## 3. Cebuano Linguistic Lexicons

In [ ]:
class CebuanoLexicons:
    NEGATORS = {'dili', 'wala', 'ayaw', 'di', 'wa', 'walay', 'walai', 'dili na', 'wala na', 'dili pa', 'wala pa'}
    CONTRAST_MARKERS = {'pero', 'apan', 'bisan pa', 'bisag', 'kondili', 'however', 'but', 'although', 'though'}
    INTENSIFIERS = {'kaayo', 'gyud', 'jud', 'grabe', 'sobra', 'dako', 'labi na', 'hilabi', 'muy', 'ayo', 'kaayu'}
    HEDGES = {'tingali', 'siguro', 'basin', 'murag', 'parang', 'daw', 'kuno', 'unta', 'basi', 'cguro'}
    POSITIVE_ANCHORS = {'nindot', 'maayo', 'gwapo', 'gwapa', 'lami', 'nice', 'nalipay', 'malipayon', 'happy', 'okay', 'ok', 'salamat', 'thank', 'thanks', 'daghang salamat'}
    NEGATIVE_ANCHORS = {'bati', 'dautan', 'pangit', 'dili maayo', 'grabe', 'naguol', 'sad', 'nabalaka', 'worried', 'kalagot', 'angry', 'kahadlok', 'scary', 'hugaw', 'dirty'}
    STOP_WORDS = {'ang', 'sa', 'og', 'ug', 'nga', 'ni', 'ko', 'ka', 'si', 'niya', 'nila', 'kami', 'kita', 'sila', 'ra', 'lang', 'man', 'na', 'pa', 'ba'}
    SCOPE_ENDERS = {',', '.', '!', '?', ';', ':', '-'}

    @classmethod
    def get_word_type(cls, word: str) -> str:
        word_lower = word.lower()
        if word_lower in cls.NEGATORS: return 'NEG'
        elif word_lower in cls.CONTRAST_MARKERS: return 'CONTRAST'
        elif word_lower in cls.INTENSIFIERS: return 'INTENS'
        elif word_lower in cls.HEDGES: return 'HEDGE'
        elif word_lower in cls.POSITIVE_ANCHORS: return 'POS_ANCHOR'
        elif word_lower in cls.NEGATIVE_ANCHORS: return 'NEG_ANCHOR'
        elif word_lower in cls.STOP_WORDS: return 'STOP'
        else: return 'NEUTRAL'

class EdgeType:
    SELF = 0; DEP = 1; NEG = 2; CONTRAST = 3; INTENS = 4; HEDGE = 5; ANCHOR = 6

    @staticmethod
    def get_weight(edge_type: int) -> float:
        weights = {0: 1.0, 1: 0.3, 2: 1.8, 3: 1.2, 4: 1.4, 5: 0.8, 6: 1.5}
        return weights.get(edge_type, 0.5)


class PhraseMatcher:
    """Greedy longest-first phrase matcher. Prevents double-counting."""
    def __init__(self, lex=None):
        if lex is None: lex = CebuanoLexicons
        self._phrases = {}
        for pset, ptype in [(lex.NEGATIVE_ANCHORS,'NEG_ANCHOR'),(lex.POSITIVE_ANCHORS,'POS_ANCHOR'),
                            (lex.NEGATORS,'NEG'),(lex.CONTRAST_MARKERS,'CONTRAST'),(lex.INTENSIFIERS,'INTENS')]:
            for phrase in pset:
                parts = phrase.lower().split()
                if len(parts) >= 2:
                    self._phrases[phrase.lower()] = (len(parts), ptype)
        self._max_ngram = max((v[0] for v in self._phrases.values()), default=1)

    def match(self, words: List[str]) -> List[Tuple[int, int, str]]:
        n = len(words); consumed = [False]*n; spans = []
        for ngram_len in range(min(self._max_ngram, n), 1, -1):
            for i in range(n - ngram_len + 1):
                if any(consumed[j] for j in range(i, i+ngram_len)): continue
                phrase = ' '.join(words[i:i+ngram_len]).lower()
                if phrase in self._phrases:
                    _, ptype = self._phrases[phrase]
                    spans.append((i, i+ngram_len, ptype))
                    for j in range(i, i+ngram_len): consumed[j] = True
        for i in range(n):
            if consumed[i]: continue
            wtype = CebuanoLexicons.get_word_type(words[i])
            if wtype not in ('NEUTRAL','STOP'):
                spans.append((i, i+1, wtype)); consumed[i] = True
        spans.sort(key=lambda x: x[0])
        return spans

print("Lexicons + EdgeType + PhraseMatcher ready")


## 4. Rule-Based Graph Builder

In [ ]:
class CebuanoGraphBuilder:
    """Phrase-aware, conditional-DEP graph builder."""
    def __init__(self):
        self.lex = CebuanoLexicons()
        self.all_anchors = self.lex.POSITIVE_ANCHORS | self.lex.NEGATIVE_ANCHORS
        self.phrase_matcher = PhraseMatcher(self.lex)

    def _is_content(self, word):
        return word.lower() not in self.lex.STOP_WORDS and word not in self.lex.SCOPE_ENDERS and len(word) > 1

    def _is_anchor(self, word):
        return word.lower() in self.all_anchors

    def _find_clause_boundaries(self, words):
        boundaries = set()
        for i, w in enumerate(words):
            if w in self.lex.SCOPE_ENDERS or w.lower() in self.lex.CONTRAST_MARKERS:
                boundaries.add(i)
        return boundaries

    def build_word_graph(self, words: List[str]) -> Tuple[np.ndarray, np.ndarray]:
        n = len(words)
        if n == 0:
            return np.zeros((0,0), dtype=np.float32), np.zeros((0,0), dtype=np.int64)
        adj = np.zeros((n,n), dtype=np.float32)
        etypes = np.zeros((n,n), dtype=np.int64)

        for i in range(n):
            adj[i,i] = EdgeType.get_weight(EdgeType.SELF)
            etypes[i,i] = EdgeType.SELF

        # Conditional DEP: skip stop-to-stop adjacency
        for i in range(n-1):
            if self._is_content(words[i]) or self._is_content(words[i+1]):
                adj[i,i+1] = adj[i+1,i] = EdgeType.get_weight(EdgeType.DEP)
                etypes[i,i+1] = etypes[i+1,i] = EdgeType.DEP

        spans = self.phrase_matcher.match(words)
        span_types = {}
        for start, end, stype in spans:
            for k in range(start, end): span_types[k] = stype

        boundaries = self._find_clause_boundaries(words)
        self._add_negation_edges(words, adj, etypes, boundaries, span_types)
        self._add_contrast_edges(words, adj, etypes, span_types)
        self._add_intensifier_edges(words, adj, etypes, span_types)
        self._add_hedge_edges(words, adj, etypes, span_types)
        self._add_anchor_edges(words, adj, etypes, span_types)
        return adj, etypes

    def _set_edge(self, adj, etypes, i, j, etype):
        w = EdgeType.get_weight(etype)
        if w > adj[i,j]:
            adj[i,j] = adj[j,i] = w
            etypes[i,j] = etypes[j,i] = etype

    def _add_negation_edges(self, words, adj, etypes, boundaries, span_types):
        n = len(words)
        for i in range(n):
            if span_types.get(i) != 'NEG': continue
            scope_end = min(i+7, n)
            for j in range(i+1, scope_end):
                if j in boundaries: scope_end = j; break
            anchor_found = False
            for j in range(i+1, scope_end):
                st = span_types.get(j, '')
                if st in ('POS_ANCHOR','NEG_ANCHOR') or self._is_anchor(words[j]):
                    self._set_edge(adj, etypes, i, j, EdgeType.NEG); anchor_found = True; break
            if not anchor_found:
                linked = 0
                for j in range(i+1, min(scope_end, i+5)):
                    if self._is_content(words[j]):
                        self._set_edge(adj, etypes, i, j, EdgeType.NEG); linked += 1
                        if linked >= 3: break

    def _add_contrast_edges(self, words, adj, etypes, span_types):
        n = len(words)
        for i in range(n):
            if span_types.get(i) != 'CONTRAST': continue
            if words[i].lower() not in self.lex.CONTRAST_MARKERS: continue
            prev_c = []
            for j in range(i-1, max(-1,i-8), -1):
                if words[j] in self.lex.SCOPE_ENDERS: break
                if self._is_content(words[j]): prev_c.append(j)
                if len(prev_c)>=4: break
            next_c = []
            for j in range(i+1, min(n,i+10)):
                if words[j] in self.lex.SCOPE_ENDERS or words[j].lower() in self.lex.CONTRAST_MARKERS: break
                if self._is_content(words[j]): next_c.append(j)
                if len(next_c)>=6: break
            for j in prev_c: self._set_edge(adj, etypes, i, j, EdgeType.CONTRAST)
            for j in next_c: self._set_edge(adj, etypes, i, j, EdgeType.CONTRAST)
            pa = [j for j in prev_c if self._is_anchor(words[j])]
            na = [j for j in next_c if self._is_anchor(words[j])]
            for p in pa[:2]:
                for q in na[:2]: self._set_edge(adj, etypes, p, q, EdgeType.CONTRAST)

    def _add_intensifier_edges(self, words, adj, etypes, span_types):
        n = len(words)
        for i in range(n):
            if span_types.get(i) != 'INTENS': continue
            for j in range(max(0,i-2), min(n,i+3)):
                if i!=j and (span_types.get(j,'') in ('POS_ANCHOR','NEG_ANCHOR') or self._is_anchor(words[j])):
                    self._set_edge(adj, etypes, i, j, EdgeType.INTENS)

    def _add_hedge_edges(self, words, adj, etypes, span_types):
        n = len(words)
        for i in range(n):
            if span_types.get(i) != 'HEDGE': continue
            linked = 0
            for j in range(i+1, min(n,i+4)):
                if self._is_content(words[j]):
                    self._set_edge(adj, etypes, i, j, EdgeType.HEDGE); linked += 1
                    if linked>=2: break

    def _add_anchor_edges(self, words, adj, etypes, span_types):
        n = len(words)
        for i in range(n):
            st = span_types.get(i, '')
            if st not in ('POS_ANCHOR','NEG_ANCHOR') and not self._is_anchor(words[i]): continue
            for j in range(max(0,i-2), min(n,i+3)):
                if i!=j and adj[i,j]==0 and self._is_content(words[j]):
                    self._set_edge(adj, etypes, i, j, EdgeType.ANCHOR)


class TokenGraphProjector:
    """All-subword projection (not first-token-only)."""
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self._has_offsets = hasattr(tokenizer, 'is_fast') and tokenizer.is_fast

    def project_graph_to_tokens(self, text, word_adj, word_etypes, max_len):
        words = text.split()
        n_words = len(words)
        if n_words == 0:
            return torch.zeros(max_len, max_len, dtype=torch.float32), torch.zeros(max_len, max_len, dtype=torch.int64)

        word_to_tokens = self._align(text, words, max_len)
        tok_adj = np.zeros((max_len, max_len), dtype=np.float32)
        tok_etypes = np.zeros((max_len, max_len), dtype=np.int64)

        for i in range(n_words):
            toks_i = word_to_tokens[i]
            for j in range(n_words):
                if word_adj[i,j] == 0: continue
                w, et = word_adj[i,j], word_etypes[i,j]
                for ti in toks_i:
                    if ti < 0 or ti >= max_len: continue
                    for tj in word_to_tokens[j]:
                        if tj < 0 or tj >= max_len: continue
                        if w > tok_adj[ti,tj]:
                            tok_adj[ti,tj] = w; tok_etypes[ti,tj] = et
        return torch.from_numpy(tok_adj), torch.from_numpy(tok_etypes)

    def _align(self, text, words, max_len):
        if self._has_offsets: return self._align_by_offsets(text, words, max_len)
        else: return self._align_by_retokenize(words, max_len)

    def _align_by_offsets(self, text, words, max_len):
        enc = self.tokenizer(text, max_length=max_len, truncation=True, return_offsets_mapping=True, add_special_tokens=True)
        offsets = enc['offset_mapping']
        word_spans, pos = [], 0
        for w in words:
            start = text.find(w, pos)
            if start == -1: start = pos
            word_spans.append((start, start+len(w))); pos = start+len(w)
        word_to_tokens = []
        for ws, we in word_spans:
            toks = [t_idx for t_idx, (ts,te) in enumerate(offsets) if not (ts==te==0) and ts < we and te > ws]
            word_to_tokens.append(toks if toks else [-1])
        return word_to_tokens

    def _align_by_retokenize(self, words, max_len):
        word_to_tokens = []; token_idx = 1
        for word in words:
            n_sub = len(self.tokenizer.tokenize(word))
            if n_sub == 0: word_to_tokens.append([-1]); continue
            toks = [token_idx+k for k in range(n_sub) if token_idx+k < max_len]
            word_to_tokens.append(toks if toks else [-1]); token_idx += n_sub
        return word_to_tokens

print("Graph builder v2 + token projector v2 ready")


## 5. Dataset

In [ ]:
import gc

# ============ SPARSE HELPERS ============

def _dense_to_sparse(tok_adj, tok_etypes):
    """Convert dense adj+etypes to sparse edge list."""
    adj_t = tok_adj if isinstance(tok_adj, torch.Tensor) else torch.from_numpy(tok_adj).float() if hasattr(tok_adj, '__array__') else torch.tensor(tok_adj, dtype=torch.float32)
    et_t = tok_etypes if isinstance(tok_etypes, torch.Tensor) else torch.from_numpy(tok_etypes).long() if hasattr(tok_etypes, '__array__') else torch.tensor(tok_etypes, dtype=torch.long)

    nz = adj_t.nonzero(as_tuple=False)
    if len(nz) == 0:
        return torch.zeros((0, 2), dtype=torch.int16), torch.zeros((0,), dtype=torch.float16), torch.zeros((0,), dtype=torch.int8)

    edge_index = nz.to(torch.int16)
    weights = adj_t[nz[:, 0], nz[:, 1]].to(torch.float16)
    etypes = et_t[nz[:, 0], nz[:, 1]].to(torch.int8)
    return edge_index, weights, etypes


def _sparse_to_dense(edge_index, edge_weight, edge_type, max_len):
    """Reconstruct dense adj+etypes from sparse edge list."""
    adj = torch.zeros(max_len, max_len, dtype=torch.float32)
    et = torch.zeros(max_len, max_len, dtype=torch.long)
    if len(edge_index) > 0:
        r, c = edge_index[:, 0].long(), edge_index[:, 1].long()
        adj[r, c] = edge_weight.float()
        et[r, c] = edge_type.long()
    return adj, et


def _process_one(text, label, tokenizer, graph_builder, graph_projector, max_len):
    """Process a single text → sparse cached item."""
    enc = tokenizer(text, max_length=max_len, padding='max_length', truncation=True, return_tensors='pt')
    words = text.split()
    word_adj, word_etypes = graph_builder.build_word_graph(words)
    tok_adj, tok_etypes = graph_projector.project_graph_to_tokens(text, word_adj, word_etypes, max_len)

    # Self-loops for non-PAD tokens
    attn_mask = enc['attention_mask'][0]
    for i in range(len(attn_mask)):
        if attn_mask[i] == 1:
            if isinstance(tok_adj, torch.Tensor):
                tok_adj[i, i] = 1.0
                tok_etypes[i, i] = EdgeType.SELF
            else:
                tok_adj[i, i] = 1.0
                tok_etypes[i, i] = EdgeType.SELF

    ei, ew, et = _dense_to_sparse(tok_adj, tok_etypes)
    return {
        'input_ids': enc['input_ids'][0],
        'attention_mask': attn_mask,
        'edge_index': ei, 'edge_weight': ew, 'edge_type': et,
        'label': label,
    }


# ============ CHUNKED CACHE BUILDER ============

def build_cache(data, split, tokenizer, graph_builder, graph_projector, config):
    """Build cache as chunk files. No aug pairs - each row is a sample."""
    cache_dir = Path(config.CACHE_DIR) / split
    cache_dir.mkdir(parents=True, exist_ok=True)
    manifest_file = cache_dir / "manifest.json"

    if config.USE_CACHE and not config.FORCE_REBUILD_CACHE and manifest_file.exists():
        with open(manifest_file) as f:
            manifest = json.load(f)
        if manifest.get('total') == len(data) and manifest.get('max_len') == config.MAX_LEN and manifest.get('cache_version', '') == config.CACHE_VERSION:
            print(f"  [{split}] LOADING cache ({config.CACHE_VERSION}): {manifest['total']:,} samples")
            return manifest

    print(f"  [{split}] BUILDING cache ({config.CACHE_VERSION}): {len(data):,} samples")
    t0 = time.time()

    chunk_buf = []
    chunk_idx = 0
    chunks_meta = []
    written = 0

    for i, item in enumerate(data):
        text = item.get('original', item.get('text', ''))
        cached = _process_one(text, item['label'], tokenizer, graph_builder, graph_projector, config.MAX_LEN)
        chunk_buf.append(cached)

        if len(chunk_buf) >= config.CHUNK_SIZE:
            fname = f"chunk_{chunk_idx:04d}.pt"
            torch.save(chunk_buf, cache_dir / fname)
            chunks_meta.append({'file': fname, 'size': len(chunk_buf), 'start': written})
            written += len(chunk_buf)
            chunk_buf = []
            chunk_idx += 1
            gc.collect()

        if (i + 1) % 5000 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta = (len(data) - i - 1) / rate
            print(f"    [{i+1:,}/{len(data):,}] {rate:.0f}/sec, ETA {eta/60:.1f}min")

    if chunk_buf:
        fname = f"chunk_{chunk_idx:04d}.pt"
        torch.save(chunk_buf, cache_dir / fname)
        chunks_meta.append({'file': fname, 'size': len(chunk_buf), 'start': written})
        written += len(chunk_buf)

    manifest = {'total': written, 'num_chunks': len(chunks_meta), 'chunks': chunks_meta, 'max_len': config.MAX_LEN, 'cache_version': config.CACHE_VERSION}
    with open(manifest_file, 'w') as f:
        json.dump(manifest, f)

    elapsed = time.time() - t0
    print(f"  [{split}] Done: {written:,} samples, {len(chunks_meta)} chunks ({elapsed/60:.1f}min)")
    gc.collect()
    return manifest


# ============ DATASET ============

class CebuanoSentimentDataset(Dataset):
    """Lazy-loads sparse chunks, reconstructs dense adj in __getitem__. No aug pairing."""

    def __init__(self, data, tokenizer, graph_builder, graph_projector, config, split='train'):
        self.split = split
        self.max_len = config.MAX_LEN
        self.cache_dir = Path(config.CACHE_DIR) / split

        manifest = build_cache(data, split, tokenizer, graph_builder, graph_projector, config)
        self.total = manifest['total']

        print(f"  [{split}] Loading {manifest['num_chunks']} chunks into RAM...")
        self.items = []
        for ci in manifest['chunks']:
            chunk = torch.load(self.cache_dir / ci['file'], weights_only=False)
            self.items.extend(chunk)
            del chunk
        gc.collect()
        print(f"  [{split}] {len(self.items):,} samples loaded")

    def __len__(self):
        return self.total

    def __getitem__(self, idx):
        d = self.items[idx]
        adj, etypes = _sparse_to_dense(d['edge_index'], d['edge_weight'], d['edge_type'], self.max_len)
        return {
            'input_ids': d['input_ids'],
            'attention_mask': d['attention_mask'],
            'adj': adj,
            'etypes': etypes,
            'label': torch.tensor(d['label'], dtype=torch.long),
        }

print("Dataset class ready (no LICR pairing)")


## 6. Model Architecture

In [ ]:
class SyntaxEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.gat_layers = nn.ModuleList([nn.MultiheadAttention(embed_dim=config.HIDDEN_DIM, num_heads=config.NUM_GAT_HEADS, dropout=config.GRAPH_DROPOUT, batch_first=True) for _ in range(2)])
        self.layer_norms = nn.ModuleList([nn.LayerNorm(config.HIDDEN_DIM) for _ in range(2)])
        self.dropout = nn.Dropout(config.GRAPH_DROPOUT)
        self.edge_embeddings = nn.Embedding(config.NUM_EDGE_TYPES, config.NUM_GAT_HEADS)

        # Meaningful edge init so graph bias starts differentiated
        with torch.no_grad():
            init_vals = torch.tensor([
                [0.0]*config.NUM_GAT_HEADS,   # SELF
                [0.1]*config.NUM_GAT_HEADS,   # DEP
                [0.8]*config.NUM_GAT_HEADS,   # NEG
                [0.6]*config.NUM_GAT_HEADS,   # CONTRAST
                [0.7]*config.NUM_GAT_HEADS,   # INTENS
                [0.3]*config.NUM_GAT_HEADS,   # HEDGE
                [0.5]*config.NUM_GAT_HEADS,   # ANCHOR
            ])
            self.edge_embeddings.weight.copy_(init_vals)

    def forward(self, hidden_states, adj, etypes, attention_mask, graph_bias_alpha=1.0):
        B, L, D = hidden_states.shape
        edge_biases = self.edge_embeddings(etypes)
        adj_expanded = adj.unsqueeze(-1)
        graph_bias = (adj_expanded * edge_biases).permute(0, 3, 1, 2)
        graph_bias = graph_bias * graph_bias_alpha
        key_mask = attention_mask.unsqueeze(1).unsqueeze(2)
        graph_bias = graph_bias.masked_fill(key_mask == 0, float('-inf'))
        x = hidden_states
        for gat, ln in zip(self.gat_layers, self.layer_norms):
            attn_out, _ = gat(x, x, x,
                              attn_mask=graph_bias.reshape(B * self.config.NUM_GAT_HEADS, L, L),
                              key_padding_mask=(attention_mask == 0))
            attn_out = torch.nan_to_num(attn_out, nan=0.0)
            x = ln(x + self.dropout(attn_out))
        return x


class AttentionOverAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.query_proj = nn.Linear(config.HIDDEN_DIM, config.HIDDEN_DIM)
        self.key_proj = nn.Linear(config.HIDDEN_DIM, config.HIDDEN_DIM)
        self.value_proj = nn.Linear(config.HIDDEN_DIM, config.HIDDEN_DIM)
        self.dropout = nn.Dropout(config.DROPOUT)
        self.scale = config.HIDDEN_DIM ** 0.5

    def forward(self, syntax_states, attention_mask):
        Q = self.query_proj(syntax_states)
        K = self.key_proj(syntax_states)
        V = self.value_proj(syntax_states)
        I = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        key_mask = attention_mask.unsqueeze(1)
        I = I.masked_fill(key_mask == 0, float('-inf'))
        alpha = F.softmax(I, dim=-1)
        alpha = torch.nan_to_num(alpha, nan=0.0)
        alpha = self.dropout(alpha)
        I_t = I.transpose(1, 2)
        beta = F.softmax(I_t, dim=-1)
        beta = torch.nan_to_num(beta, nan=0.0)
        beta_avg = beta.mean(dim=1, keepdim=True)
        attn_logits = torch.bmm(alpha.transpose(1, 2), beta_avg.transpose(1, 2)).squeeze(-1)
        pad_mask = (attention_mask == 0)
        attn_logits = attn_logits.masked_fill(pad_mask, float('-inf'))
        final_attn = F.softmax(attn_logits, dim=-1)
        final_attn = torch.nan_to_num(final_attn, nan=0.0)
        aoa_output = torch.bmm(final_attn.unsqueeze(1), V).squeeze(1)
        return aoa_output, final_attn


class CebuanoSentimentModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.encoder = AutoModel.from_pretrained(config.MODEL_NAME)
        self.syntax_encoder = SyntaxEncoder(config)
        self.aoa = AttentionOverAttention(config)
        self.cls_dropout = nn.Dropout(config.CLS_DROPOUT)

        # ── UPGRADE 2: Gated Fusion Mechanism ─────────────────────────────
        # Replaces the old torch.cat([cls, aoa]) + MLP fusion block.
        # A learned sigmoid gate dynamically balances semantic (CLS) and
        # syntactic (AoA) representations, preventing feature collision on
        # the neutral class caused by naive concatenation.
        #
        #   z_gate  = sigmoid( W · [cls ; aoa] )          shape: (B, H)
        #   fused   = z_gate * cls + (1 - z_gate) * aoa   shape: (B, H)
        #
        # The gate is computed from the full concatenated context but the
        # output dimensionality stays at hidden_size (not hidden_size * 2),
        # so the downstream classifier is unchanged.
        self.fusion_gate = nn.Linear(config.HIDDEN_DIM * 2, config.HIDDEN_DIM)

        # Classifier receives hidden_size features (not hidden_size * 2)
        self.classifier = nn.Linear(config.HIDDEN_DIM, config.NUM_CLASSES)
        self.graph_classifier = nn.Sequential(
            nn.Linear(config.HIDDEN_DIM, config.HIDDEN_DIM // 2),
            nn.ReLU(),
            nn.Dropout(config.DROPOUT),
            nn.Linear(config.HIDDEN_DIM // 2, config.NUM_CLASSES)
        )
        self.projection_head = nn.Sequential(
            nn.Linear(config.HIDDEN_DIM, config.HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(config.HIDDEN_DIM, 128)
        )

    def forward(self, input_ids, attention_mask, adj, etypes,
                training=True, graph_bias_alpha=1.0, cls_mask_prob=0.0):
        encoder_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = encoder_out.last_hidden_state
        cls_token = hidden_states[:, 0, :]
        syntax_states = self.syntax_encoder(hidden_states, adj, etypes, attention_mask, graph_bias_alpha=graph_bias_alpha)

        # ── UPGRADE 3: ENABLE_SA_AOA toggle ───────────────────────────────
        # When False, bypass the dual-perspective AoA filter entirely and
        # use the mean of TD-GAT output states as the syntactic summary.
        if self.config.ENABLE_SA_AOA:
            aoa_output, attn_weights = self.aoa(syntax_states, attention_mask)
        else:
            # Ablation: mean-pool syntax_states across the sequence dimension
            aoa_output = syntax_states.mean(dim=1)
            B, L, _ = syntax_states.shape
            # Uniform attention weights (valid tensor, no NaNs)
            attn_weights = torch.ones(B, L, device=syntax_states.device) / L

        if training:
            cls_token = self.cls_dropout(cls_token)
            if cls_mask_prob > 0:
                keep = (torch.rand(cls_token.size(0), 1, device=cls_token.device) >= cls_mask_prob).float()
                cls_token = cls_token * keep

        # ── UPGRADE 2: Gated Fusion (replaces torch.cat + MLP) ────────────
        z_gate = torch.sigmoid(self.fusion_gate(torch.cat([cls_token, aoa_output], dim=-1)))
        fused_features = (z_gate * cls_token) + ((1 - z_gate) * aoa_output)

        logits = self.classifier(fused_features)
        graph_logits = self.graph_classifier(aoa_output)
        scl_features = F.normalize(self.projection_head(fused_features), p=2, dim=1)
        return {
            'logits': logits, 'graph_logits': graph_logits, 'scl_features': scl_features,
            'attention_weights': attn_weights, 'fused_features': fused_features, 'aoa_output': aoa_output
        }

print("Model ready (Gated Fusion + SA-AoA toggle + graph_bias_alpha + CLS masking + edge init)")


## 7. Loss Functions

In [ ]:
class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        similarity = torch.matmul(features, features.T) / self.temperature
        labels = labels.unsqueeze(1)
        mask_pos = (labels == labels.T).float()
        mask_pos.fill_diagonal_(0)
        logits_max, _ = similarity.max(dim=1, keepdim=True)
        logits = similarity - logits_max.detach()
        exp_logits = torch.exp(logits)
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-8)
        mask_pos_sum = torch.clamp(mask_pos.sum(dim=1), min=1.0)
        mean_log_prob_pos = (mask_pos * log_prob).sum(dim=1) / mask_pos_sum
        return -mean_log_prob_pos.mean()


class EntropyRegularizationLoss(nn.Module):
    """Entropy of AoA attention weights. Clamp before log to prevent NaN."""
    def forward(self, attention_weights):
        # TASK 4: clamp before log
        p = attention_weights.clamp(min=1e-8)
        entropy = -(p * torch.log(p)).sum(dim=-1)
        return -entropy.mean()

print("Loss functions ready (NaN-safe entropy)")


## 8. Training Functions

In [ ]:
def pred_entropy_from_logits(logits):
    probs = F.softmax(logits, dim=-1)
    return -(probs * (probs + 1e-12).log()).sum(dim=-1).mean()


def train_epoch(model, loader, optimizer, scheduler, config, scl_loss_fn, entropy_loss_fn, scaler, device,
                gamma_entropy=None, cls_mask_prob=0.0):
    if gamma_entropy is None: gamma_entropy = config.GAMMA_ENTROPY_END
    model.train()
    total_loss = 0
    ce_total = graph_ce_total = scl_total = attn_ent_total = pred_ent_total = 0
    all_preds, all_labels = [], []
    nan_skips = 0

    ce_loss_fn = nn.CrossEntropyLoss(
        weight=config.CLASS_WEIGHTS.to(device) if config.CLASS_WEIGHTS is not None else None,
        label_smoothing=config.LABEL_SMOOTHING)
    accum = config.GRAD_ACCUM_STEPS
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="Training")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        adj = batch['adj'].to(device)
        etypes = batch['etypes'].to(device)
        labels = batch['label'].to(device)

        with autocast('cuda'):
            outputs = model(input_ids, attention_mask, adj, etypes,
                            training=True, graph_bias_alpha=config.GRAPH_BIAS_ALPHA, cls_mask_prob=cls_mask_prob)
            if torch.isnan(outputs['logits']).any() or torch.isnan(outputs['attention_weights']).any():
                nan_skips += 1
                if nan_skips <= 5: print(f"  [!] NaN at step {step}")
                optimizer.zero_grad(set_to_none=True); continue

            ce_loss = ce_loss_fn(outputs['logits'], labels)
            graph_ce_loss = ce_loss_fn(outputs['graph_logits'], labels)

            # Base: Cross-Entropy + Graph Auxiliary Cross-Entropy (always active)
            loss = ce_loss + config.LAMBDA_GRAPH_AUX * graph_ce_loss

            # ── UPGRADE 3: ENABLE_SCL toggle ──────────────────────────────
            # Add Supervised Contrastive Loss only when the flag is True.
            if config.ENABLE_SCL:
                scl_loss = scl_loss_fn(outputs['scl_features'].float(), labels)
                loss = loss + config.ALPHA_SCL * scl_loss
            else:
                scl_loss = torch.tensor(0.0)

            # ── UPGRADE 3: ENABLE_ENTROPY toggle ──────────────────────────
            # Add Entropy Regularization loss only when the flag is True.
            if config.ENABLE_ENTROPY:
                attn_entropy = entropy_loss_fn(outputs['attention_weights'].float())
                loss = loss + gamma_entropy * attn_entropy
            else:
                attn_entropy = torch.tensor(0.0)

            # ── UPGRADE 1: Gradient Accumulation ──────────────────────────
            # Divide by GRAD_ACCUM_STEPS so the effective loss magnitude
            # matches what a single large-batch step would produce.
            loss = loss / accum

        scaler.scale(loss).backward()
        if (step + 1) % accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(); scheduler.step()

        with torch.no_grad():
            p_ent = pred_entropy_from_logits(outputs['logits'].float())
        total_loss += loss.item() * accum
        ce_total += ce_loss.item(); graph_ce_total += graph_ce_loss.item()
        scl_total += scl_loss.item(); attn_ent_total += attn_entropy.item()
        pred_ent_total += p_ent.item()
        preds = outputs['logits'].argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())

    n = max(len(loader) - nan_skips, 1)
    if nan_skips > 0: print(f"  [!] NaN skipped: {nan_skips}")
    avg_ce = ce_total / n; avg_ae = attn_ent_total / n
    return {
        'loss': total_loss/n, 'ce': avg_ce, 'graph_ce': graph_ce_total/n, 'scl': scl_total/n,
        'attn_entropy': avg_ae, 'pred_entropy': pred_ent_total/n,
        'gamma_entropy': gamma_entropy, 'cls_mask_prob': cls_mask_prob,
        'weighted_entropy': gamma_entropy * avg_ae,
        'entropy_ce_ratio': (gamma_entropy * avg_ae) / max(avg_ce, 1e-8),
        'accuracy': accuracy_score(all_labels, all_preds) if all_preds else 0.0,
        'macro_f1': f1_score(all_labels, all_preds, average='macro') if all_preds else 0.0,
    }


def evaluate(model, loader, config, device):
    model.eval()
    all_preds, all_labels = [], []
    attn_ent_total = pred_ent_total = 0; n_batches = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            with autocast('cuda'):
                outputs = model(batch['input_ids'].to(device), batch['attention_mask'].to(device),
                                batch['adj'].to(device), batch['etypes'].to(device),
                                training=False, graph_bias_alpha=config.GRAPH_BIAS_ALPHA, cls_mask_prob=0.0)
            logits = outputs['logits']
            if torch.isnan(logits).any(): logits = torch.nan_to_num(logits, nan=0.0)
            aw = outputs['attention_weights'].clamp(min=1e-8)
            attn_ent_total += -(aw * aw.log()).sum(-1).mean().item()
            pred_ent_total += pred_entropy_from_logits(logits.float()).item()
            n_batches += 1
            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_labels.extend(batch['label'].cpu().numpy())

    p, r, f1c, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, zero_division=0)
    n = max(n_batches, 1)
    return {
        'accuracy': accuracy_score(all_labels, all_preds),
        'macro_f1': f1_score(all_labels, all_preds, average='macro'),
        'precision_neg': p[0], 'precision_pos': p[1], 'precision_neu': p[2],
        'recall_neg': r[0], 'recall_pos': r[1], 'recall_neu': r[2],
        'f1_neg': f1c[0], 'f1_pos': f1c[1], 'f1_neu': f1c[2],
        'attn_entropy': attn_ent_total / n, 'pred_entropy': pred_ent_total / n,
    }

print("Training functions ready")


## 9. Load Data

In [ ]:
print("Loading data...")
with open(config.DATA_PATH, 'r', encoding='utf-8') as f:
    if config.DATA_PATH.endswith('.jsonl'):
        data = [json.loads(line) for line in f]
    else:
        data = json.load(f)

print(f"Loaded {len(data)} samples")

unique_ids = list(set(item['id'] for item in data))
train_ids, temp_ids = train_test_split(unique_ids, test_size=0.3, random_state=SEED)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=SEED)

train_data = [item for item in data if item['id'] in train_ids]
val_data = [item for item in data if item['id'] in val_ids]
test_data = [item for item in data if item['id'] in test_ids]

print(f"Split - Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

labels = [item['label'] for item in train_data]
label_counts = np.bincount(labels)
total = len(labels)
class_weights = torch.FloatTensor([total / (config.NUM_CLASSES * c) for c in label_counts])
config.CLASS_WEIGHTS = class_weights
print(f"Class distribution: {label_counts}")
print(f"Class weights: {class_weights.numpy()}")

## 10. Create Datasets and DataLoaders

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME, use_fast=True)
graph_builder = CebuanoGraphBuilder()
graph_projector = TokenGraphProjector(tokenizer)

print("Creating datasets...")
train_dataset = CebuanoSentimentDataset(train_data, tokenizer, graph_builder, graph_projector, config, split='train')
val_dataset = CebuanoSentimentDataset(val_data, tokenizer, graph_builder, graph_projector, config, split='val')
test_dataset = CebuanoSentimentDataset(test_data, tokenizer, graph_builder, graph_projector, config, split='test')

# ── UPGRADE 1: Optimized DataLoader pipeline ──────────────────────────────
# num_workers=2  — two background workers prefetch batches while the GPU
#                  computes, eliminating the CPU→GPU data-starvation bottleneck.
# pin_memory=True — pre-pins host memory so CUDA can DMA-transfer directly,
#                   cutting host-to-device transfer latency.
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=config.BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=config.BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset):,} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset):,} samples, {len(val_loader)} batches")
print(f"Test:  {len(test_dataset):,} samples, {len(test_loader)} batches")

# Sanity check
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
for k in ['input_ids', 'attention_mask', 'adj', 'etypes', 'label']:
    print(f"  {k}: {batch[k].shape} {batch[k].dtype}")


## 11. Initialize Model

In [ ]:
print("Initializing model...")
model = CebuanoSentimentModel(config).to(device)

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    print(f"Wrapping model in DataParallel ({n_gpus} GPUs)")
    model = nn.DataParallel(model)
else:
    print(f"Single GPU mode")

# Helper to access base model (unwrap DataParallel)
def get_base_model(m):
    return m.module if hasattr(m, 'module') else m

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"GPUs: {n_gpus}")


## 12. Training

In [ ]:
base_model = get_base_model(model)
encoder_params = list(base_model.encoder.parameters())
other_params = [p for n, p in base_model.named_parameters() if not n.startswith('encoder.')]
optimizer = torch.optim.AdamW([
    {'params': encoder_params, 'lr': config.ENCODER_LR},
    {'params': other_params, 'lr': config.LEARNING_RATE}
], weight_decay=config.WEIGHT_DECAY)

total_steps = (len(train_loader) // config.GRAD_ACCUM_STEPS) * config.EPOCHS
warmup_steps = int(total_steps * config.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

scl_loss_fn = SupervisedContrastiveLoss(config.TEMPERATURE_SCL)
entropy_loss_fn = EntropyRegularizationLoss()
scaler = GradScaler()

history = {
    'train_loss': [], 'train_acc': [], 'train_f1': [], 'val_acc': [], 'val_f1': [],
    'ce': [], 'graph_ce': [], 'scl': [],
    'attn_entropy': [], 'pred_entropy': [], 'val_attn_entropy': [], 'val_pred_entropy': [],
    'gamma_entropy': [], 'weighted_entropy': [], 'entropy_ce_ratio': [], 'cls_mask_prob': [],
}
best_val_f1 = 0
best_model_state = None

print("\n" + "="*80)
print("STARTING TRAINING (10 epochs) — XLM-RoBERTa backbone")
print(f"  Loss = CE + {config.LAMBDA_GRAPH_AUX}*GraphCE + {config.ALPHA_SCL}*SCL + gamma*AttnEntropy")
print(f"  GRAPH_BIAS_ALPHA = {config.GRAPH_BIAS_ALPHA}")
print(f"  CLS masking: {config.CLS_MASK_START} -> {config.CLS_MASK_END}")
print(f"  Entropy: {config.GAMMA_ENTROPY_START} -> {config.GAMMA_ENTROPY_END} (warmup {config.ENTROPY_WARMUP_EPOCHS} ep)")
print(f"  GPUs: {torch.cuda.device_count()}")
print("="*80)

for epoch in range(config.EPOCHS):
    # Entropy schedule
    if epoch < config.ENTROPY_WARMUP_EPOCHS:
        gamma_entropy = config.GAMMA_ENTROPY_START
    else:
        frac = (epoch - config.ENTROPY_WARMUP_EPOCHS) / max(1, config.EPOCHS - config.ENTROPY_WARMUP_EPOCHS)
        gamma_entropy = config.GAMMA_ENTROPY_START + (config.GAMMA_ENTROPY_END - config.GAMMA_ENTROPY_START) * frac

    # CLS masking schedule: linear decay from START to END over all epochs
    cls_frac = epoch / max(1, config.EPOCHS - 1)
    cls_mask_prob = config.CLS_MASK_START + (config.CLS_MASK_END - config.CLS_MASK_START) * cls_frac

    print(f"\nEpoch {epoch+1}/{config.EPOCHS}  [gamma={gamma_entropy:.3f}  alpha={config.GRAPH_BIAS_ALPHA}  cls_mask={cls_mask_prob:.3f}]")
    print("-" * 80)

    tm = train_epoch(model, train_loader, optimizer, scheduler, config,
                     scl_loss_fn, entropy_loss_fn, scaler, device,
                     gamma_entropy=gamma_entropy, cls_mask_prob=cls_mask_prob)
    vm = evaluate(model, val_loader, config, device)

    print(f"Train — Loss:{tm['loss']:.4f}  Acc:{tm['accuracy']:.4f}  F1:{tm['macro_f1']:.4f}")
    print(f"  CE:{tm['ce']:.4f}  GraphCE:{tm['graph_ce']:.4f}  SCL:{tm['scl']:.4f}")
    print(f"  AttnEnt:{tm['attn_entropy']:.4f}  PredEnt:{tm['pred_entropy']:.4f}  g*AEnt:{tm['weighted_entropy']:.4f}  ratio:{tm['entropy_ce_ratio']:.4f}")
    print(f"Val   — Acc:{vm['accuracy']:.4f}  F1:{vm['macro_f1']:.4f}  AttnEnt:{vm['attn_entropy']:.4f}  PredEnt:{vm['pred_entropy']:.4f}")
    print(f"  F1: Neg:{vm['f1_neg']:.4f}  Pos:{vm['f1_pos']:.4f}  Neu:{vm['f1_neu']:.4f}")

    if vm['macro_f1'] > best_val_f1:
        best_val_f1 = vm['macro_f1']
        best_model_state = {
            'epoch': epoch, 'val_f1': best_val_f1, 'config': config,
            'model_state_dict': get_base_model(model).state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }
        print(f"  *** New best model! F1: {best_val_f1:.4f} ***")

    history['train_loss'].append(tm['loss']); history['train_acc'].append(tm['accuracy']); history['train_f1'].append(tm['macro_f1'])
    history['val_acc'].append(vm['accuracy']); history['val_f1'].append(vm['macro_f1'])
    history['ce'].append(tm['ce']); history['graph_ce'].append(tm['graph_ce']); history['scl'].append(tm['scl'])
    history['attn_entropy'].append(tm['attn_entropy']); history['pred_entropy'].append(tm['pred_entropy'])
    history['val_attn_entropy'].append(vm['attn_entropy']); history['val_pred_entropy'].append(vm['pred_entropy'])
    history['gamma_entropy'].append(gamma_entropy); history['weighted_entropy'].append(tm['weighted_entropy'])
    history['entropy_ce_ratio'].append(tm['entropy_ce_ratio']); history['cls_mask_prob'].append(cls_mask_prob)

checkpoint_path = Path(config.OUTPUT_DIR) / "best_model.pt"
torch.save(best_model_state, checkpoint_path)
print(f"\nBest model saved to {checkpoint_path}")
print(f"Best Val F1: {best_val_f1:.4f}")


## 13. Final Test Evaluation

In [ ]:
print("\n" + "="*80)
print("FINAL TEST EVALUATION")
print("="*80)

get_base_model(model).load_state_dict(best_model_state['model_state_dict'])
test_metrics = evaluate(model, test_loader, config, device)

print(f"\nTest Results:")
print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
print(f"  Macro F1: {test_metrics['macro_f1']:.4f}")
print(f"\nPer-Class Metrics:")
print(f"  NEGATIVE - P: {test_metrics['precision_neg']:.4f}, R: {test_metrics['recall_neg']:.4f}, F1: {test_metrics['f1_neg']:.4f}")
print(f"  POSITIVE - P: {test_metrics['precision_pos']:.4f}, R: {test_metrics['recall_pos']:.4f}, F1: {test_metrics['f1_pos']:.4f}")
print(f"  NEUTRAL  - P: {test_metrics['precision_neu']:.4f}, R: {test_metrics['recall_neu']:.4f}, F1: {test_metrics['f1_neu']:.4f}")


## 14. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_f1'], label='Train F1', marker='o')
axes[1].plot(history['val_f1'], label='Val F1', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro F1')
axes[1].set_title('F1 Score')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/training_curves.png', dpi=150)
plt.show()
print(f"Saved to {config.OUTPUT_DIR}/training_curves.png")

## 15. Done

In [ ]:
print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print(f"\nFinal Results:")
print(f"  Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"  Test Macro F1: {test_metrics['macro_f1']:.4f}")
print(f"\nModel saved to: {config.OUTPUT_DIR}/best_model.pt")
print(f"\nUse the inference notebook to make predictions!")

## 16. Evaluation & Diagnostic Metrics (DAS + ECE)
**Dependency Awareness Score** — real graph vs identity graph (self-loops only).
**Expected Calibration Error** — temperature fit on **validation** only, applied to **test** (no leakage).

In [ ]:
from collections import defaultdict
SELF_TYPE_ID = EdgeType.SELF

def build_identity_graph(B, seq_len, attention_mask, device):
    adj = torch.eye(seq_len, dtype=torch.float32, device=device).unsqueeze(0).expand(B, -1, -1).clone()
    etypes = torch.zeros(B, seq_len, seq_len, dtype=torch.long, device=device)
    idx = torch.arange(seq_len, device=device)
    etypes[:, idx, idx] = SELF_TYPE_ID
    m = attention_mask.float(); mask_2d = m.unsqueeze(2) * m.unsqueeze(1)
    return adj * mask_2d, etypes * mask_2d.long()

@torch.no_grad()
def collect_graph_modes(model, loader, device, alpha=None):
    if alpha is None: alpha = config.GRAPH_BIAS_ALPHA
    model.eval()
    out = {'labels': [], 'logits_r': [], 'logits_i': []}
    for batch in tqdm(loader, desc="DAS eval"):
        ids, mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)
        adj, et = batch['adj'].to(device), batch['etypes'].to(device)
        B, L = ids.size(0), ids.size(1)
        with autocast('cuda'):
            lr = model(ids, mask, adj, et, training=False, graph_bias_alpha=alpha, cls_mask_prob=0.0)['logits'].float()
        lr = torch.nan_to_num(lr, nan=0.0)
        id_adj, id_et = build_identity_graph(B, L, mask, device)
        with autocast('cuda'):
            li = model(ids, mask, id_adj, id_et, training=False, graph_bias_alpha=alpha, cls_mask_prob=0.0)['logits'].float()
        li = torch.nan_to_num(li, nan=0.0)
        out['labels'].append(batch['label']); out['logits_r'].append(lr.cpu()); out['logits_i'].append(li.cpu())
    out['labels'] = torch.cat(out['labels']).numpy()
    out['logits_r'] = torch.cat(out['logits_r']); out['logits_i'] = torch.cat(out['logits_i'])
    return out

@torch.no_grad()
def alpha_sweep(model, loader, device, alphas=[0, 1.0, 2.0, 4.0, 8.0]):
    model.eval()
    results = {a: [] for a in alphas}; ident_results = {a: [] for a in alphas}; all_labels = []
    for batch in tqdm(loader, desc="Alpha sweep"):
        ids, mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)
        adj, et = batch['adj'].to(device), batch['etypes'].to(device)
        B, L = ids.size(0), ids.size(1); all_labels.extend(batch['label'].numpy())
        id_adj, id_et = build_identity_graph(B, L, mask, device)
        for a in alphas:
            with autocast('cuda'):
                lr = model(ids, mask, adj, et, training=False, graph_bias_alpha=a, cls_mask_prob=0.0)['logits'].float()
                li = model(ids, mask, id_adj, id_et, training=False, graph_bias_alpha=a, cls_mask_prob=0.0)['logits'].float()
            results[a].append(torch.nan_to_num(lr, nan=0.0).cpu())
            ident_results[a].append(torch.nan_to_num(li, nan=0.0).cpu())
    all_labels = np.array(all_labels); eps = 1e-8; sweep_out = {}
    for a in alphas:
        lr = torch.cat(results[a]); li = torch.cat(ident_results[a])
        pr, pi = F.softmax(lr,1).numpy(), F.softmax(li,1).numpy()
        preds_r, preds_i = lr.argmax(1).numpy(), li.argmax(1).numpy()
        p, q = np.clip(pr,eps,1), np.clip(pi,eps,1)
        kl = float(np.mean(np.sum(p*np.log(p/q),1)))
        m = 0.5*(p+q); js = float(np.mean(0.5*np.sum(p*np.log(p/m),1)+0.5*np.sum(q*np.log(q/m),1)))
        dl2 = float(np.mean(np.sqrt(np.sum((lr.numpy()-li.numpy())**2,1))))
        sweep_out[a] = {
            'f1_real': float(f1_score(all_labels,preds_r,average='macro')),
            'f1_ident': float(f1_score(all_labels,preds_i,average='macro')),
            'das_f1': float(f1_score(all_labels,preds_r,average='macro'))-float(f1_score(all_labels,preds_i,average='macro')),
            'flip': float(np.mean(preds_r!=preds_i)), 'kl': kl, 'js': js, 'dl2': dl2,
        }
    return sweep_out

def calc_metrics(labels, preds):
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    return {'accuracy': float(accuracy_score(labels,preds)), 'macro_f1': float(f1_score(labels,preds,average='macro')),
            'per_class': {'negative':{'p':float(p[0]),'r':float(r[0]),'f1':float(f1[0])},
                          'positive':{'p':float(p[1]),'r':float(r[1]),'f1':float(f1[1])},
                          'neutral':{'p':float(p[2]),'r':float(r[2]),'f1':float(f1[2])}}}

def expected_calibration_error(probs, labels, n_bins=15):
    confs, preds = np.max(probs,1), np.argmax(probs,1)
    correct = (preds==labels).astype(float); edges = np.linspace(0,1,n_bins+1); ece = 0.0
    for lo,hi in zip(edges[:-1],edges[1:]):
        m = (confs>lo)&(confs<=hi)
        if m.sum()==0: continue
        ece += m.sum()*abs(correct[m].mean()-confs[m].mean())
    return float(ece/len(labels))

def fit_temperature(val_logits, val_labels, lr=0.01, max_iter=100):
    T = torch.nn.Parameter(torch.tensor(1.5)); lg = val_logits.clone().detach().float()
    lb = torch.tensor(val_labels, dtype=torch.long); nll = nn.CrossEntropyLoss()
    opt = torch.optim.LBFGS([T], lr=lr, max_iter=max_iter)
    def closure(): opt.zero_grad(); loss = nll(lg/T, lb); loss.backward(); return loss
    opt.step(closure); return float(T.item())

def apply_temperature(logits_np, T):
    sc = logits_np/T; e = np.exp(sc-sc.max(1,keepdims=True)); return e/e.sum(1,keepdims=True)

# ==================================================================
print("\n" + "="*80)
print("EVALUATION & DIAGNOSTICS")
print("="*80)

if best_model_state is not None:
    get_base_model(model).load_state_dict(best_model_state['model_state_dict'])
    print(f"Best model loaded (epoch {best_model_state['epoch']+1}, F1={best_model_state['val_f1']:.4f})\n")

# Alpha sweep
print("--- Alpha Sweep (real vs identity per alpha) ---")
sweep = alpha_sweep(model, test_loader, device, alphas=[0, 1.0, 2.0, 4.0, 8.0])
print(f"\n  {'alpha':<6} {'F1_real':>8} {'F1_ident':>9} {'DAS_F1':>8} {'Flip%':>7} {'KL':>9} {'JS':>9} {'dL2':>7}")
print("  "+"-"*70)
for a in [0,1.0,2.0,4.0,8.0]:
    s = sweep[a]
    print(f"  {a:<6.1f} {s['f1_real']:>8.4f} {s['f1_ident']:>9.4f} {s['das_f1']:>+8.4f} {s['flip']*100:>6.1f}% {s['kl']:>9.4f} {s['js']:>9.6f} {s['dl2']:>7.3f}")

# DAS at training alpha
print(f"\n--- DAS at alpha={config.GRAPH_BIAS_ALPHA} ---")
test_out = collect_graph_modes(model, test_loader, device)
pr_r, pd_r = F.softmax(test_out['logits_r'],1).numpy(), test_out['logits_r'].argmax(1).numpy()
pr_i, pd_i = F.softmax(test_out['logits_i'],1).numpy(), test_out['logits_i'].argmax(1).numpy()
mr, mi = calc_metrics(test_out['labels'], pd_r), calc_metrics(test_out['labels'], pd_i)
das_f1 = mr['macro_f1']-mi['macro_f1']; flip = float(np.mean(pd_r!=pd_i))
eps = 1e-8; p,q = np.clip(pr_r,eps,1), np.clip(pr_i,eps,1)
avg_kl = float(np.mean(np.sum(p*np.log(p/q),1)))
m = 0.5*(p+q); avg_js = float(np.mean(0.5*np.sum(p*np.log(p/m),1)+0.5*np.sum(q*np.log(q/m),1)))
dl2 = float(np.mean(np.sqrt(np.sum((test_out['logits_r'].numpy()-test_out['logits_i'].numpy())**2,1))))
print(f"  DAS_F1: {das_f1:+.4f}  FlipRate: {flip:.4f} ({flip*100:.1f}%)  KL: {avg_kl:.6f}  JS: {avg_js:.6f}  dL2: {dl2:.4f}")
print(f"\n  {'Metric':<20} {'Real':>10} {'Identity':>10} {'Delta':>10}")
print("  "+"-"*53)
print(f"  {'Accuracy':<20} {mr['accuracy']:>10.4f} {mi['accuracy']:>10.4f} {mr['accuracy']-mi['accuracy']:>+10.4f}")
print(f"  {'Macro F1':<20} {mr['macro_f1']:>10.4f} {mi['macro_f1']:>10.4f} {das_f1:>+10.4f}")
for c in ['negative','positive','neutral']:
    fr, fi = mr['per_class'][c]['f1'], mi['per_class'][c]['f1']
    print(f"    F1 {c:<14} {fr:>10.4f} {fi:>10.4f} {fr-fi:>+10.4f}")

# ECE
print(f"\n--- Calibration ---")
val_out = collect_graph_modes(model, val_loader, device)
ece_b = expected_calibration_error(pr_r, test_out['labels'])
T = fit_temperature(val_out['logits_r'], val_out['labels'])
ece_a = expected_calibration_error(apply_temperature(test_out['logits_r'].numpy(), T), test_out['labels'])
ece_i = expected_calibration_error(pr_i, test_out['labels'])
print(f"  T*={T:.4f}  ECE: before={ece_b:.4f} after={ece_a:.4f} identity={ece_i:.4f}")

# Entropy history
if history.get('attn_entropy'):
    print(f"\n--- Entropy & CLS Mask History ---")
    print(f"  {'Ep':<4} {'gamma':>6} {'cls_m':>6} {'AttnEnt':>9} {'PredEnt':>9} {'g*AEnt':>8} {'CE':>8} {'Ratio':>8}")
    for ep in range(len(history['attn_entropy'])):
        cm = history['cls_mask_prob'][ep]
        print(f"  {ep+1:<4} {history['gamma_entropy'][ep]:>6.3f} {cm:>6.3f} {history['attn_entropy'][ep]:>9.4f} {history['pred_entropy'][ep]:>9.4f} {history['weighted_entropy'][ep]:>8.4f} {history['ce'][ep]:>8.4f} {history['entropy_ce_ratio'][ep]:>8.4f}")

# Save
save_dir = Path(config.OUTPUT_DIR)
with open(save_dir/"dependency_awareness.json",'w') as f:
    json.dump({'DAS_F1':das_f1,'FlipRate':flip,'AvgKL':avg_kl,'AvgJS':avg_js,'dL2':dl2,'alpha':config.GRAPH_BIAS_ALPHA,'f1_real':mr['macro_f1'],'f1_ident':mi['macro_f1']},f,indent=2)
with open(save_dir/"alpha_sweep.json",'w') as f:
    json.dump({str(k):v for k,v in sweep.items()},f,indent=2)
with open(save_dir/"calibration.json",'w') as f:
    json.dump({'T':T,'ECE_before':ece_b,'ECE_after':ece_a,'ECE_identity':ece_i},f,indent=2)
with open(save_dir/"eval_real.json",'w') as f: json.dump(mr,f,indent=2)
with open(save_dir/"eval_identity.json",'w') as f: json.dump(mi,f,indent=2)
if history.get('attn_entropy'):
    with open(save_dir/"entropy_diagnostics.json",'w') as f:
        json.dump({k:history[k] for k in ['attn_entropy','pred_entropy','ce','gamma_entropy','weighted_entropy','entropy_ce_ratio','cls_mask_prob'] if k in history},f,indent=2)
print(f"\nArtifacts saved to {save_dir}/")


## 17. Download Checkpoint

In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

src  = Path(config.OUTPUT_DIR) / "best_model.pt"
dest = Path("/kaggle/working/best_model_xlmroberta.pt")

if src.exists():
    shutil.copy(src, dest)
    size_mb = dest.stat().st_size / (1024 ** 2)
    print(f"Checkpoint : {src}")
    print(f"Size       : {size_mb:.1f} MB")
    print(f"Epoch      : {best_model_state['epoch'] + 1}   Val F1: {best_model_state['val_f1']:.4f}")
    print("")
    display(FileLink("best_model_xlmroberta.pt", result_html_prefix="Download: "))
else:
    print(f"Not found: {src}")
    print("Run the training cell first.")
